In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 149.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "speakleash/Bielik-11B-v2.5-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print(f"Ładuję model: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
print("Model gotowy do pracy!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.1 MB/s eta 0:00:00
Ładuję model: speakleash/Bielik-11B-v2.5-Instruct...


config.json:   0%|          | 0.00/709 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/27.1k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/164 [00:00<?, ?B/s]

Model gotowy do pracy!


# Analiza tekstu literackiego (streszczenie)

In [ ]:
!pip install -q pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 106.7 MB/s eta 0:00:00


In [ ]:
import fitz  # PyMuPDF
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# 2. Funkcja przygotowująca dokumenty JSON (Standard RAG)
def prepare_rag_documents(file_path, pages_to_extract):
    documents = []
    doc_pdf = fitz.open(file_path)

    for i, pg_num in enumerate(pages_to_extract):
        if pg_num <= len(doc_pdf):
            page = doc_pdf[pg_num - 1]
            text = page.get_text().strip()

            # Struktura słownika zgodnie z Twoim wymogiem
            doc_entry = {
                "doc_id": i,
                "title": f"Janko Muzykant - Strona {pg_num}",
                "content": text[:4000] # Ograniczenie długości chunka
            }
            documents.append(doc_entry)

    doc_pdf.close()
    return json.dumps(documents, ensure_ascii=False)

In [ ]:
# 3. Parametry
pdf_file = "/content/janko-muzykant.pdf"
strony = [2, 3]

# Generowanie kontekstu
rag_context_json = prepare_rag_documents(pdf_file, strony)

In [ ]:
# 4. Prompt ChatML zoptymalizowany pod Bielika
messages = [
    {
        "role": "system",
        "content": (
            "Jesteś pomocnym asystentem i ekspertem od literatury. "
            "Twoim zadaniem jest analiza tekstów dostarczonych w formacie JSON i tworzenie "
            "rzetelnych streszczeń. Zawsze opieraj się wyłącznie na dostarczonym materiale."
        )
    },
    {
        "role": "user",
        "content": f"Oto dokumenty źródłowe w formacie JSON:\n{rag_context_json}\n\nPrzeanalizuj te fragmenty 'Janka Muzykanta' i napisz ich zwięzłe streszczenie po polsku."
    }
]

encoded_input = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True
)
input_ids = encoded_input['input_ids'].to("cuda")

In [ ]:
# 5. Generowanie
print(f"Bielik analizuje tekst przy użyciu metody RAG...\n")

outputs = model.generate(
    input_ids,
    max_new_tokens=1000,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.1
)

summary = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)

print("--- STRESZCZENIE RAG (BIELIK) ---")
print(summary)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Bielik analizuje tekst przy użyciu metody RAG...

--- STRESZCZENIE RAG (BIELIK) ---
**Strona 2:**
Janko rodzi się jako słabe dziecko. Kumy obawiają się o jego życie i chrzczą go, nadając imię Jan. Mimo choroby, chłopiec przeżywa i wyrasta na wątłego, ale żywiołowego chłopca. Od najmłodszych lat fascynuje go muzyka – słyszy ją w naturze, w borze, w wietrze, a nawet w wodzie. Jego matka, biedna komornica, często bije go i nazywa „odmieńcem”. Janko nie interesuje się pracą, ale marzy o graniu na skrzypcach.

**Strona 3:**
Janko uwielbia muzykę i słucha jej wszędzie – w borze, w polu, w wiosce. Często wymyka się nocą pod karczmę, by podsłuchiwać muzyków. Marzy o własnych skrzypkach, ale nie ma ich skąd wziąć. Robi więc prowizoryczne instrumenty z gonta i włosienia końskiego, ale brzmią one słabo. Chłopiec głoduje, ale nie przestaje marzyć o prawdziwych skrzypcach. Pewnej nocy, gdy we dworze nikogo nie ma, Janko zakrada się do kredensu, by dotknąć upragnionych skrzypiec.

**Podsumowanie:**


--- STRESZCZENIE RAG (BIELIK) ---

**Strona 2:**
Janko rodzi się jako słabe dziecko. Kumy obawiają się o jego życie i chrzczą go, nadając imię Jan. Mimo choroby, chłopiec przeżywa i wyrasta na wątłego, ale żywiołowego chłopca. Od najmłodszych lat fascynuje go muzyka – słyszy ją w naturze, w borze, w wietrze, a nawet w wodzie. Jego matka, biedna komornica, często bije go i nazywa „odmieńcem”. Janko nie interesuje się pracą, ale marzy o graniu na skrzypcach.

**Strona 3:**
Janko uwielbia muzykę i słucha jej wszędzie – w borze, w polu, w wiosce. Często wymyka się nocą pod karczmę, by podsłuchiwać muzyków. Marzy o własnych skrzypkach, ale nie ma ich skąd wziąć. Robi więc prowizoryczne instrumenty z gonta i włosienia końskiego, ale brzmią one słabo. Chłopiec głoduje, ale nie przestaje marzyć o prawdziwych skrzypcach. Pewnej nocy, gdy we dworze nikogo nie ma, Janko zakrada się do kredensu, by dotknąć upragnionych skrzypiec.

**Podsumowanie:**
„Janko Muzykant” to historia biednego, wiejskiego chłopca, który od dziecka pasjonuje się muzyką. Jego marzenia o graniu na skrzypcach pozostają niespełnione, a życie wypełnia bieda i ciężka praca. Opowiadanie ukazuje tragiczny los dziecka, które nie ma szans na realizację swoich pasji.

In [ ]:
# 2. RAG Document Preparation Function
def prepare_rag_documents(file_path, pages_to_extract):
    documents = []
    doc_pdf = fitz.open(file_path)

    for i, pg_num in enumerate(pages_to_extract):
        if pg_num <= len(doc_pdf):
            page = doc_pdf[pg_num - 1]
            text = page.get_text().strip()

            # Dictionary structure for RAG
            doc_entry = {
                "doc_id": i,
                "title": f"The Hobbit - Page {pg_num}",
                "content": text[:4000] # Chunk length limit
            }
            documents.append(doc_entry)

    doc_pdf.close()
    return json.dumps(documents, ensure_ascii=False)

In [ ]:
# 3. Execution Parameters
pdf_file = "/content/the-hobbit-1.pdf"
pages = [67, 68] # Adjust pages as needed

# Context Generation
rag_context_json = prepare_rag_documents(pdf_file, pages)

In [ ]:
# 4. English ChatML Prompt for Bielik
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful assistant and a literary expert. "
            "Your task is to analyze texts provided in JSON format and create "
            "reliable summaries. Always base your response strictly on the provided material."
        )
    },
    {
        "role": "user",
        "content": f"Here are the source documents in JSON format:\n{rag_context_json}\n\nAnalyze these fragments of 'The Hobbit' and write a concise summary of them in English."
    }
]

# Apply Chat Template
# Note: returning 'input_ids' directly to handle typical Bielik/Transformers output
encoded_input = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True
)
input_ids = encoded_input['input_ids'].to("cuda")

In [ ]:
# 5. Generation
print(f"Bielik is analyzing the text using RAG method...\n")

outputs = model.generate(
    input_ids,
    max_new_tokens=1000,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.1
)

# Decoding
summary = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)

print("--- ENGLISH RAG SUMMARY (BIELIK) ---")
print(summary)

Bielik is analyzing the text using RAG method...

--- ENGLISH RAG SUMMARY (BIELIK) ---
In Chapter V of "The Hobbit," Bilbo Baggins wakes up disoriented and alone in a pitch-black tunnel after falling off a goblin cart. He cannot see or hear anything and feels only the stone floor beneath him. After crawling around and failing to find any signs of goblins or dwarves, Bilbo discovers a small metal ring, which he puts in his pocket without thinking. Feeling lost and miserable, he sits down and thinks of his home and missed meals.

Later, Bilbo remembers his pipe and finds tobacco in his pouch, but no matches. He then recalls his elven dagger from the trolls, which gives him courage. Deciding to press forward, Bilbo trots cautiously through the tunnel with his sword drawn and one hand on the wall. Despite being in a tight spot, Bilbo's knowledge of tunneling and resilience help him navigate the darkness.


--- ENGLISH RAG SUMMARY (BIELIK) ---

In Chapter V of "The Hobbit," Bilbo Baggins wakes up disoriented and alone in a pitch-black tunnel after falling off a goblin cart. He cannot see or hear anything and feels only the stone floor beneath him. After crawling around and failing to find any signs of goblins or dwarves, Bilbo discovers a small metal ring, which he puts in his pocket without thinking. Feeling lost and miserable, he sits down and thinks of his home and missed meals.

Later, Bilbo remembers his pipe and finds tobacco in his pouch, but no matches. He then recalls his elven dagger from the trolls, which gives him courage. Deciding to press forward, Bilbo trots cautiously through the tunnel with his sword drawn and one hand on the wall. Despite being in a tight spot, Bilbo's knowledge of tunneling and resilience help him navigate the darkness.

In [ ]:
# 2. Przygotowanie dokumentów JSON (Struktura RAG)
def prepare_rag_documents(file_path, pages_to_extract):
    documents = []
    doc_pdf = fitz.open(file_path)

    for i, pg_num in enumerate(pages_to_extract):
        if pg_num <= len(doc_pdf):
            page = doc_pdf[pg_num - 1]
            text = page.get_text().strip()

            doc_entry = {
                "doc_id": i,
                "title": f"Преступление и наказание - Страница {pg_num}",
                "content": text[:4000]
            }
            documents.append(doc_entry)

    doc_pdf.close()
    return json.dumps(documents, ensure_ascii=False)

In [ ]:
# 3. Parametry
pdf_file = "/content/priestuplieniie_i_nakazaniie_-_fiedor_mikhailovich_dostoievskii.pdf"
pages = [12, 13] # Wybrane strony do analizy

rag_context_json = prepare_rag_documents(pdf_file, pages)

In [ ]:
# 4. Prompt w języku rosyjskim
messages = [
    {
        "role": "system",
        "content": (
            "Вы — эксперт по русской классической литературе. Ваша задача — проанализировать "
            "фрагменты текста, представленные в формате JSON, и составить подробное, "
            "профессиональное краткое содержание на РУССКОМ языке."
        )
    },
    {
        "role": "user",
        "content": f"Вот исходные документы в формате JSON:\n{rag_context_json}\n\nНа основе этих страниц напишите краткое резюме событий и описание душевного состояния главного героя на русском языке."
    }
]

# Apply Chat Template
# Note: returning 'input_ids' directly to handle typical Bielik/Transformers output
encoded_input = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True
)
input_ids = encoded_input['input_ids'].to("cuda")

In [ ]:
# 5. Generation
print(f"Bielik is analyzing the text using RAG method...\n")

outputs = model.generate(
    input_ids,
    max_new_tokens=1000,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.1
)

# Decoding
summary = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)

print("--- RUSSIAN RAG SUMMARY ---")
print(summary)

Bielik is analyzing the text using RAG method...

--- RUSSIAN RAG SUMMARY ---
**Резюме событий:**

На страницах 12-13 романа Ф.М. Достоевского "Преступление и наказание" описываются встречи Раскольникова с Маarmеладовым. Маarmеладов рассказывает о своей жизни, страданиях и попытках найти смысл. Он упоминает о своей жене Катерине Ивановне, которая является образованной и гордой женщиной, но живёт в нищете. Маarmеладов говорит о том, как они переехали в столицу, где он потерял работу и начал пить. Раскольников, наблюдая за Маarmеладовым, пытается понять причины его поведения и страданий.

**Описание душевного состояния Раскольникова:**

Раскольников наблюдает за Маarmеладовым с чувством сочувствия и одновременно непонимания. Он задумывается над причинами поведения Маarmеладова и своим собственным положением. Раскольников чувствует, что Маarmеладов страдает, но не может или не хочет изменить свою жизнь. Это заставляет Раскольникова задуматься над собственными ценностями и моралью. Он начи

--- RUSSIAN RAG SUMMARY ---

**Резюме событий:**

На страницах 12-13 романа Ф.М. Достоевского "Преступление и наказание" описываются встречи Раскольникова с Маarmеладовым. Маarmеладов рассказывает о своей жизни, страданиях и попытках найти смысл. Он упоминает о своей жене Катерине Ивановне, которая является образованной и гордой женщиной, но живёт в нищете. Маarmеладов говорит о том, как они переехали в столицу, где он потерял работу и начал пить. Раскольников, наблюдая за Маarmеладовым, пытается понять причины его поведения и страданий.

**Описание душевного состояния Раскольникова:**

Раскольников наблюдает за Маarmеладовым с чувством сочувствия и одновременно непонимания. Он задумывается над причинами поведения Маarmеладова и своим собственным положением. Раскольников чувствует, что Маarmеладов страдает, но не может или не хочет изменить свою жизнь. Это заставляет Раскольникова задуматься над собственными ценностями и моралью. Он начинает сомневаться в своей идее о превосходстве разума над чувствами.

**Краткое резюме:**

Встречи Раскольникова с Маarmеладовым раскрывают трагическую судьбу последнего, его страдания и попытки найти смысл в жизни. Раскольников, наблюдая за Маarmеладовым, задумывается над собственными ценностями и моралью, что приближает его к кризису и пониманию собственной вины.

# Analiza tekstu technicznego

Analiza tekstu technicznego polegała na sprawdzeniu jak dobrze model rozumie restrykcje zawarte w tekstu i skomplikowane definicje

In [ ]:
# 2. Funkcja przygotowująca dokumenty JSON (Standard RAG)
def prepare_rag_documents(file_path, pages_to_extract):
    documents = []
    doc_pdf = fitz.open(file_path)

    for i, pg_num in enumerate(pages_to_extract):
        if pg_num <= len(doc_pdf):
            page = doc_pdf[pg_num - 1]
            text = page.get_text().strip()

            doc_entry = {
                "doc_id": i,
                "title": f"Hager Instrukcja - Strona {pg_num}",
                "content": text[:4000]
            }
            documents.append(doc_entry)

    doc_pdf.close()
    return json.dumps(documents, ensure_ascii=False)

In [ ]:
# 3. Parametry wejściowe
pdf_file = "/content/19PL004_HAAG_BR_Handbuch_Schutz_vers_PL_2019.pdf"

In [ ]:
# --- ZADANIE 2: Zasady bezpieczeństwa ---
strony_safety = [23]
context_safety = prepare_rag_documents(pdf_file, strony_safety)

prompt_safety = [
    {
        "role": "system",
        "content": "Jesteś ekspertem w dziedzinie instalacji elektrycznych. Twoim zadaniem jest analiza polskiej dokumentacji technicznej Hager. Odpowiadaj wyłącznie na podstawie dostarczonych danych JSON."
    },
    {
        "role": "user",
        "content": f"Oto treść instrukcji Hager:\n{context_safety}\n\nZadanie: Wypunktuj 5 najważniejszych zasad bezpieczeństwa opisanych w tym fragmencie. Skup się na tym, czego kategorycznie zabrania instrukcja."
    }
]

# --- ZADANIE 3: Wyjaśnienie terminologii ---
strony_defs = [4]
context_defs = prepare_rag_documents(pdf_file, strony_defs)

prompt_defs = [
    {
        "role": "system",
        "content": "Jesteś ekspertem w dziedzinie instalacji elektrycznych. Twoim zadaniem jest analiza polskiej dokumentacji technicznej Hager. Odpowiadaj wyłącznie na podstawie dostarczonych danych JSON."
    },
    {
        "role": "user",
        "content": f"Oto treść instrukcji Hager:\n{context_defs}\n\nZadanie: Zidentyfikuj 3 skróty techniczne w tekście (np. RCD, RCCB, RCBO) i wyjaśnij ich znaczenie na podstawie dokumentu."
    }
]

In [ ]:
# 4. Funkcja do generowania odpowiedzi
def generate_response(messages):
    encoded_input = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    )
    input_ids = encoded_input['input_ids'].to("cuda")

    outputs = model.generate(
        input_ids,
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.1
    )
    return tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)

In [ ]:
# 5. Wykonanie analizy
print("--- BIELIK: WYNIK ZADANIA 2 (BEZPIECZEŃSTWO) ---")
print(generate_response(prompt_safety))

print("\n" + "="*50 + "\n")

print("--- BIELIK: WYNIK ZADANIA 3 (DEFINICJE) ---")
print(generate_response(prompt_defs))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--- BIELIK: WYNIK ZADANIA 2 (BEZPIECZEŃSTWO) ---
Na podstawie analizy dostarczonego fragmentu instrukcji Hager, poniżej przedstawiam 5 najważniejszych zasad bezpieczeństwa, które kategorycznie zabraniają pewnych działań:

1. **Zakaz stosowania przewodów płytszych niż 6 mm od powierzchni ściany** w pomieszczeniach z wanną lub prysznicem.
2. **Zakaz używania ręcznych urządzeń elektrycznych o prądzie znamionowym > 32 A** na placach budowy.
3. **Zakaz stosowania gniazd elektrycznych o prądzie znamionowym > 32 A** w rolnictwie i ogrodnictwie.
4. **Zakaz stosowania gniazd elektrycznych o prądzie znamionowym > 32 A** w przyczepach kempingowych i domach przewoźnych.
5. **Zakaz stosowania gniazd elektrycznych o prądzie znamionowym > 32 A** w portach jachtowych i podobnych miejscach.

Te zasady mają na celu minimalizację ryzyka porażenia prądem w specyficznych warunkach środowiskowych.


--- BIELIK: WYNIK ZADANIA 3 (DEFINICJE) ---
1. **RCD (Residual Current Device)** – Wyłącznik różnicowoprądowy

--- BIELIK: WYNIK ZADANIA (BEZPIECZEŃSTWO) ---

Na podstawie analizy dostarczonego fragmentu instrukcji Hager, poniżej przedstawiam 5 najważniejszych zasad bezpieczeństwa, które kategorycznie zabraniają pewnych działań:

1. **Zakaz stosowania przewodów płytszych niż 6 mm od powierzchni ściany** w pomieszczeniach z wanną lub prysznicem.
2. **Zakaz używania ręcznych urządzeń elektrycznych o prądzie znamionowym > 32 A** na placach budowy.
3. **Zakaz stosowania gniazd elektrycznych o prądzie znamionowym > 32 A** w rolnictwie i ogrodnictwie.
4. **Zakaz stosowania gniazd elektrycznych o prądzie znamionowym > 32 A** w przyczepach kempingowych i domach przewoźnych.
5. **Zakaz stosowania gniazd elektrycznych o prądzie znamionowym > 32 A** w portach jachtowych i podobnych miejscach.

Te zasady mają na celu minimalizację ryzyka porażenia prądem w specyficznych warunkach środowiskowych.

==================================================

--- BIELIK: WYNIK ZADANIA (DEFINICJE) ---
1. **RCD (Residual Current Device)** – Wyłącznik różnicowoprądowy, który chroni przed porażeniem prądem elektrycznym poprzez wykrywanie niewielkich różnic między prądem wpływającym a wypływającym z obwodu.

2. **RCCB (Residual Current Circuit Breaker)** – Bezpiecznik różnicowoprądowy, odmiana wyłącznika różnicowoprądowego, która dodatkowo pełni funkcję zabezpieczenia nadprądowego.

3. **RCBO (Residual Current Circuit Breaker with Overcurrent Protection)** – Kombinacja bezpiecznika różnicowoprądowego i nadprądowego w jednym urządzeniu, często stosowana w domowych instalacjach elektrycznych.

In [ ]:
# 2. Funkcja przygotowująca dokumenty JSON (Standard RAG)
def prepare_rag_documents(file_path, pages_to_extract):
    documents = []
    doc_pdf = fitz.open(file_path)

    for i, pg_num in enumerate(pages_to_extract):
        if pg_num <= len(doc_pdf):
            page = doc_pdf[pg_num - 1]
            text = page.get_text().strip()

            doc_entry = {
                "doc_id": i,
                "title": f"ICAO Instructions - Page {pg_num}",
                "content": text[:4000]
            }
            documents.append(doc_entry)

    doc_pdf.close()
    return json.dumps(documents, ensure_ascii=False)

In [ ]:
# 3. Parametry wejściowe
pdf_file = "/content/english instruction.pdf"

In [ ]:
# --- TASK 2: Safety Procedures ---
safety_pages = [24]
context_safety = prepare_rag_documents(pdf_file, safety_pages)

prompt_safety = [
    {
        "role": "system",
        "content": "You are a specialized aviation safety inspector. Provide professional and precise answers based ONLY on the provided JSON data."
    },
    {
        "role": "user",
        "content": f"Here is the ICAO technical documentation:\n{context_safety}\n\nTask: List the 5 most critical safety rules or prohibitions described in this section. Focus specifically on what is strictly forbidden."
    }
]

# --- TASK 3: Technical Terminology ---
def_pages = [34]
context_defs = prepare_rag_documents(pdf_file, def_pages)

prompt_defs = [
    {
        "role": "system",
        "content": "You are a specialized aviation safety inspector. Provide professional and precise answers based ONLY on the provided JSON data."
    },
    {
        "role": "user",
        "content": f"Here is the ICAO technical documentation:\n{context_defs}\n\nTask: Identify 3 technical abbreviations or terms from the text and explain their meaning based on the document."
    }
]


In [ ]:
# 4. Funkcja do generowania odpowiedzi
def generate_response(messages):
    encoded_input = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    )
    input_ids = encoded_input['input_ids'].to("cuda")

    outputs = model.generate(
        input_ids,
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.1
    )
    return tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)

In [ ]:
# 5. Wykonanie analizy
print("--- BIELIK: TASK 2 RESULT (SAFETY & PROHIBITIONS) ---")
print(generate_response(prompt_safety))

print("\n" + "="*50 + "\n")

print("--- BIELIK: TASK 3 RESULT (DEFINITIONS & TERMINOLOGY) ---")
print(generate_response(prompt_defs))

--- BIELIK: TASK 2 RESULT (SAFETY & PROHIBITIONS) ---
Based on the provided ICAO Instructions, here are the five most critical safety rules or prohibitions described in the section, focusing specifically on what is strictly forbidden:

1. **Prohibition on Carrying Dangerous Goods Without Approval**: Dangerous goods cannot be carried on an aircraft unless they are approved by the operator (Section 1.1.5.1.a.1).

2. **Securing Gas Cylinders Specifically Manufactured for Containing and Transporting Gases**: Gas cylinders must be manufactured specifically for the purpose of containing and transporting the particular gas (Section 1.1.5.1.a.1).

3. **Keeping Equipment Containing Wet Cell Batteries Upright**: Equipment containing wet cell batteries must be kept and secured in an upright position to prevent spillage of the electrolyte (Section 1.1.5.1.a.2).

4. **Restrictions on Dropping Substances for Agricultural, Horticultural, Forestry, Avalanche Control, Ice Jam Control, and Landslide Cle

--- BIELIK: TASK 2 RESULT (SAFETY & PROHIBITIONS) ---

Based on the provided ICAO Instructions, here are the five most critical safety rules or prohibitions described in the section, focusing specifically on what is strictly forbidden:

1. **Prohibition on Carrying Dangerous Goods Without Approval**: Dangerous goods cannot be carried on an aircraft unless they are approved by the operator (Section 1.1.5.1.a.1).

2. **Securing Gas Cylinders Specifically Manufactured for Containing and Transporting Gases**: Gas cylinders must be manufactured specifically for the purpose of containing and transporting the particular gas (Section 1.1.5.1.a.1).

3. **Keeping Equipment Containing Wet Cell Batteries Upright**: Equipment containing wet cell batteries must be kept and secured in an upright position to prevent spillage of the electrolyte (Section 1.1.5.1.a.2).

4. **Restrictions on Dropping Substances for Agricultural, Horticultural, Forestry, Avalanche Control, Ice Jam Control, and Landslide Clearance Activities**: Dangerous goods can only be dropped in connection with these activities if they are authorized by the appropriate authorities and meet specific terms and conditions (Section 1.1.5.1.c).

5. **Vehicles in Vehicle Ferry Operations Must Meet Specific Requirements**: Vehicles carried in aircraft designed or modified for vehicle ferry operations must be secured in an upright position, fuel tanks must be filled to prevent spillage, and adequate ventilation rates must be maintained (Sections 1.1.5.1.e.1, e.2, e.3, and e.4).

==================================================

--- BIELIK: TASK 3 RESULT (DEFINITIONS & TERMINOLOGY) ---
1. **GHS**: The fourth revised edition of the Globally Harmonized System of Classification and Labelling of Chemicals, published by the United Nations as document ST/SG/AC.10/30/Rev. 4.
   *Explanation*: GHS is a global system for classifying and labeling chemicals to ensure worker safety. It helps in identifying hazards associated with chemical substances and mixtures.

2. **UN Model Recommendations**: A set of guidelines and standards for the safe transportation of dangerous goods by various modes of transport, including road, rail, sea, and air.
   *Explanation*: These recommendations provide a unified approach to the classification, packaging, and transport of hazardous materials worldwide, ensuring safety and reducing risks during transit.

3. **IBCs (Intermediate Bulk Containers)**: Rigid or flexible portable packaging, other than those specified in Part 6 of the UN Model Recommendations, designed for mechanical handling and resistant to the stresses produced in handling and transport, as determined by tests.
   *Explanation*: IBCs are large containers used for storing and transporting liquids, gases, or solids in bulk quantities. They are specifically designed to handle and transport hazardous materials safely, adhering to strict standards and tests.

In [ ]:
# 2. Funkcja przygotowująca dokumenty JSON (Standard RAG)
def prepare_rag_documents(file_path, pages_to_extract):
    documents = []
    doc_pdf = fitz.open(file_path)

    for i, pg_num in enumerate(pages_to_extract):
        if pg_num <= len(doc_pdf):
            page = doc_pdf[pg_num - 1]
            text = page.get_text().strip()

            doc_entry = {
                "doc_id": i,
                "title": f"Техническая инструкция - Страница {pg_num}",
                "content": text[:4000]
            }
            documents.append(doc_entry)

    doc_pdf.close()
    return json.dumps(documents, ensure_ascii=False)

In [ ]:
# 3. Parametry wejściowe
pdf_file = "/content/geokniga-tehnicheskayainstrukciyapoprovedeniyugeofizicheskihissledovaniyvskvazhinah.pdf"

In [ ]:
# --- ZADANIE 2: Bezpieczeństwo i awarie ---
safety_pages = [290]
context_safety = prepare_rag_documents(pdf_file, safety_pages)

prompt_safety = [
    {
        "role": "system",
        "content": "Вы — ведущий инженер-геофизик. Ваша задача — анализировать техническую документацию. Отвечайте ТОЛЬКО на основе предоставленных данных JSON на РУССКОМ языке."
    },
    {
        "role": "user",
        "content": f"Вот данные из технической инструкции:\n{context_safety}\n\nЗадание: Перечислите 5 наиболее важных правил техники безопасности или действий при авариях. Сосредоточьтесь на том, что категорически запрещено инструкцией."
    }
]

# --- ZADANIE 3: Terminologia i skróty ---
def_pages = [15]
context_defs = prepare_rag_documents(pdf_file, def_pages)

prompt_defs = [
    {
        "role": "system",
        "content": "Вы — ведущий инженер-геофизик. Ваша задача — анализировать техническую документацию. Отвечайте ТОЛЬКО на основе предоставленных данных JSON на РУССКОМ языке."
    },
    {
        "role": "user",
        "content": f"Вот данные из технической инструкции:\n{context_defs}\n\nЗадание: Найдите в тексте 3 технических сокращения (например, ГИС, КС, ПС) и объясните их значение на основе предоставленного документа."
    }
]


In [ ]:
# 4. Funkcja do generowania odpowiedzi (zintegrowana z szablonem Bielika)
def generate_response(messages):
    encoded_input = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    )
    input_ids = encoded_input['input_ids'].to("cuda")

    outputs = model.generate(
        input_ids,
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.1
    )
    return tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)

In [ ]:
# 5. Wykonanie analizy
print("--- BIELIK: РЕЗУЛЬТАТ ЗАДАНИЯ 2 (БЕЗОПАСНОСТЬ) ---")
print(generate_response(prompt_safety))

print("\n" + "="*50 + "\n")

print("--- BIELIK: РЕЗУЛЬТАТ ЗАДАНИЯ 3 (ТЕРМИНЫ И СОКРАЩЕНИЯ) ---")
print(generate_response(prompt_defs))

--- BIELIK: РЕЗУЛЬТАТ ЗАДАНИЯ 2 (БЕЗОПАСНОСТЬ) ---
1. **Немедленное сообщение о сложных авариях**: Инструкция требует немедленно сообщать о сложных авариях (включая те, которые связаны с человеческими жертвами) в вышестоящую организацию, органам горно-технического надзора, техническому инспектору профсоюза и органам милиции.

2. **Обязательная экспертиза при разногласиях**: В случае разногласий между представителями геофизического предприятия и организацией, проводящей бурение, в определении причин аварии стороны должны прибегнуть к технической экспертизе вышестоящих организаций. Решение экспертизы является обязательным для обеих сторон.

3. **Составление акта о любых авариях**: На все происшедшие в скважинах аварии, независимо от их характера, включая случаи оставления грузов, зондов и т. д., составляется акт главным инженером предприятия, начальником партии (отряда), представителем организации, проводящей буровые работы (форма 1).

4. **Причина аварии и виновники**: В акте о происшед

--- BIELIK: РЕЗУЛЬТАТ ЗАДАНИЯ 2 (БЕЗОПАСНОСТЬ) ---

1. **Немедленное сообщение о сложных авариях**: Инструкция требует немедленно сообщать о сложных авариях (включая те, которые связаны с человеческими жертвами) в вышестоящую организацию, органам горно-технического надзора, техническому инспектору профсоюза и органам милиции.

2. **Обязательная экспертиза при разногласиях**: В случае разногласий между представителями геофизического предприятия и организацией, проводящей бурение, в определении причин аварии стороны должны прибегнуть к технической экспертизе вышестоящих организаций. Решение экспертизы является обязательным для обеих сторон.

3. **Составление акта о любых авариях**: На все происшедшие в скважинах аварии, независимо от их характера, включая случаи оставления грузов, зондов и т. д., составляется акт главным инженером предприятия, начальником партии (отряда), представителем организации, проводящей буровые работы (форма 1).

4. **Причина аварии и виновники**: В акте о происшедшей аварии должна быть установлена причина аварии и виновники ее.

5. **Рассмотрение сложных аварий комиссией**: Сложные аварии расследуются комиссией в составе главного инженера предприятия, начальника партии (отряда), представителя органов горно-технического надзора, представителя буровой организации, а по мере надобности — представителя автоинспекции и местной пожарной охраны. Комиссия обязана установить причины аварии, конкретных виновников и рекомендовать меры воздействия в отношении них.

==================================================

--- BIELIK: РЕЗУЛЬТАТ ЗАДАНИЯ 3 (ТЕРМИНЫ И СОКРАЩЕНИЯ) ---
1. **ПС (Полная схема)**  
   Сокращение "ПС" используется в контексте записи кривой Полной схемы (кривая ПС). Описано в пункте 4а) и 4б) документа:  
   *"...стандартный электрический каротаж* ;...запись кривой ПС."*  
   Значение: Кривая Полной схемы — это графическое представление результатов электрических каротажей, где отражаются изменения удельного сопротивления горных пород на разных глубинах.

2. **БКЗ (Боковое каротажное зондирование)**  
   Сокращение "БКЗ" используется в пунктах 4а), 4б), 5а) и 6а) документа:  
   *"...боковое каротажное зондирование (БКЗ)**;...БКЗ;...БКЗ."*  
   Значение: Боковое каротажное зондирование — метод геофизических исследований, который производится с помощью боковых скважин (боковых ответвлений) от основной скважины. Используется для получения дополнительной информации о свойствах горных пород в боковых направлениях.

3. **ГК (Гамма-каротаж)**  
   Сокращение "ГК" упоминается в пунктах 4в), 5б), 6б) и 6в) документа:  
   *"...гамма-каротаж (ГК);...гамма-каротаж (ГК);...гамма-каротаж (ГК);...гамма-каротаж (ГК)."*  
   Значение: Гамма-каротаж — метод геофизических исследований, основанный на измерении радиоактивного излучения (гамма-излучения) горных пород. Используется для определения типа горных пород (например, песчаник, глина), их плотности и других физических свойств.

## **Iłościowa analiza porównawcza**

**6 prostych zadań**

english prompt

In [ ]:
import pandas as pd
import torch
import time

# 1. Load dataset
df = pd.read_csv("/content/tiktok_instagram_global_100countries.csv")

print("Dataset shape:", df.shape)
print(df.columns.tolist())
df.head()

Dataset shape: (10000, 23)
['user_id', 'country', 'age', 'year', 'internet_penetration', 'mental_health_support_index', 'gdp_index', 'youth_population_ratio', 'baseline_addiction_pressure', 'tiktok_minutes_daily', 'instagram_minutes_daily', 'night_usage_ratio', 'scroll_velocity', 'addiction_pressure', 'attention_span_score', 'dopamine_dependency_score', 'impulsivity_index', 'sleep_hours', 'sleep_quality_index', 'ASI', 'MHRI', 'addiction_score', 'addiction_level']


,user_id,country,age,year,internet_penetration,mental_health_support_index,gdp_index,youth_population_ratio,baseline_addiction_pressure,tiktok_minutes_daily,...,addiction_pressure,attention_span_score,dopamine_dependency_score,impulsivity_index,sleep_hours,sleep_quality_index,ASI,MHRI,addiction_score,addiction_level
0,1,Philippines,59,2050,88.895513,64.347080,0.312067,0.310389,64.762693,242.853130,...,69.981792,77.479304,27.024835,60.044856,6.561778,7.912360,100.0,40.460626,63.512418,High
1,2,Netherlands,23,2035,57.950292,76.257045,0.460320,0.293134,37.592646,98.599951,...,56.862382,78.852713,25.376744,81.769114,9.400197,2.292106,100.0,79.424028,62.688372,High
2,3,Laos,26,2060,40.325805,55.846514,0.893658,0.183055,22.589595,127.221573,...,41.817693,83.573012,19.712386,99.613065,7.810417,2.308761,100.0,88.262729,59.856193,High
3,4,Bulgaria,50,2046,43.838044,28.390576,0.661523,0.155038,57.093084,140.528514,...,70.811984,83.437675,19.874790,86.935944,7.020033,4.512440,100.0,70.905771,59.937395,High
4,5,Zambia,24,2045,43.749943,81.413755,0.275186,0.457115,57.653485,187.748696,...,71.990949,71.993561,33.607727,45.275020,7.900621,4.265014,100.0,51.312439,66.803864,High


In [ ]:
correct_answers = {
    "E1": df.loc[df["addiction_score"].idxmax(), "country"],
    "E2": round(df["addiction_score"].mean(), 2),
    "E3": df.groupby("year")["tiktok_minutes_daily"].mean().idxmax(),
    "E4": int((df["addiction_level"] == "High").sum()),
    "E5": round(df["sleep_hours"].min(), 2),
    "E6": df.groupby("country")["instagram_minutes_daily"].mean().idxmax()
}

correct_answers

{'E1': 'Ghana',
 'E2': np.float64(58.55),
 'E3': np.int64(2044),
 'E4': 8680,
 'E5': 3.0,
 'E6': 'UK'}

In [ ]:
simple_tasks = [
    {
        "task_id": "E1",
        "difficulty": "easy",
        "question": "Which country has the highest addiction_score in the provided table?",
        "data": df[["country", "addiction_score"]]
            .sort_values("addiction_score", ascending=False)
            .head(30)
    },
    {
        "task_id": "E2",
        "difficulty": "easy",
        "question": "What is the average addiction_score in the provided table? Round the answer to two decimal places.",
        "data": pd.DataFrame({
            "average_addiction_score": [round(df["addiction_score"].mean(), 2)]
        })
    },
    {
        "task_id": "E3",
        "difficulty": "easy",
        "question": "Which year has the highest average tiktok_minutes_daily in the provided table?",
        "data": df.groupby("year", as_index=False)["tiktok_minutes_daily"]
            .mean()
            .sort_values("tiktok_minutes_daily", ascending=False)
    },
    {
        "task_id": "E4",
        "difficulty": "easy",
        "question": "How many observations have addiction_level equal to High?",
        "data": pd.DataFrame({
            "addiction_level": ["High"],
            "number_of_observations": [int((df["addiction_level"] == "High").sum())]
        })
    },
    {
        "task_id": "E5",
        "difficulty": "easy",
        "question": "What is the minimum sleep_hours value in the provided table? Round the answer to two decimal places.",
        "data": pd.DataFrame({
            "minimum_sleep_hours": [round(df["sleep_hours"].min(), 2)]
        })
    },
    {
        "task_id": "E6",
        "difficulty": "easy",
        "question": "Which country has the highest average instagram_minutes_daily in the provided table?",
        "data": df.groupby("country", as_index=False)["instagram_minutes_daily"]
            .mean()
            .sort_values("instagram_minutes_daily", ascending=False)
            .head(30)
    }
]

In [ ]:
def generate_response(messages, max_new_tokens=100):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Изменили тут: temperature=0.0 убран
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.time()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip(), round(end_time - start_time, 3)

In [ ]:
results = []

for task in simple_tasks:
    table_text = task["data"].to_string(index=False)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a data analyst. "
                "Use only the provided table. "
                "Return only the final answer. "
                "Do not explain your reasoning."
            )
        },
        {
            "role": "user",
            "content": f"""
Here is the table:

{table_text}

Task:
{task["question"]}
"""
        }
    ]

    answer, response_time = generate_response(messages)

    results.append({
        "model": "Bielik-11B",
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "question": task["question"],
        "model_answer": answer,
        "correct_answer": correct_answers[task["task_id"]],
        "response_time_sec": response_time
    })

results_df = pd.DataFrame(results)
results_df

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

,model,task_id,difficulty,question,model_answer,correct_answer,response_time_sec
0,Bielik-11B,E1,easy,Which country has the highest addiction_score ...,Ghana,Ghana,0.588
1,Bielik-11B,E2,easy,What is the average addiction_score in the pro...,58.55,58.55,0.853
2,Bielik-11B,E3,easy,Which year has the highest average tiktok_minu...,2015,2044,0.742
3,Bielik-11B,E4,easy,How many observations have addiction_level equ...,8680,8680,0.690
4,Bielik-11B,E5,easy,What is the minimum sleep_hours value in the p...,3.00,3.0,0.692
5,Bielik-11B,E6,easy,Which country has the highest average instagra...,UK,UK,0.310


polish prompt

In [ ]:
simple_tasks_pl = [
    {
        "task_id": "E1",
        "difficulty": "easy",
        "language": "PL",
        "question": "Który kraj ma najwyższą wartość addiction_score w podanej tabeli?",
        "data": df[["country", "addiction_score"]]
            .sort_values("addiction_score", ascending=False)
            .head(30)
    },
    {
        "task_id": "E2",
        "difficulty": "easy",
        "language": "PL",
        "question": "Jaka jest średnia wartość addiction_score w podanej tabeli? Zaokrąglij odpowiedź do dwóch miejsc po przecinku.",
        "data": pd.DataFrame({
            "average_addiction_score": [round(df["addiction_score"].mean(), 2)]
        })
    },
    {
        "task_id": "E3",
        "difficulty": "easy",
        "language": "PL",
        "question": "W którym roku średnia wartość tiktok_minutes_daily była najwyższa w podanej tabeli?",
        "data": df.groupby("year", as_index=False)["tiktok_minutes_daily"]
            .mean()
            .sort_values("tiktok_minutes_daily", ascending=False)
    },
    {
        "task_id": "E4",
        "difficulty": "easy",
        "language": "PL",
        "question": "Ile obserwacji ma addiction_level równy High?",
        "data": pd.DataFrame({
            "addiction_level": ["High"],
            "number_of_observations": [int((df["addiction_level"] == "High").sum())]
        })
    },
    {
        "task_id": "E5",
        "difficulty": "easy",
        "language": "PL",
        "question": "Jaka jest minimalna wartość sleep_hours w podanej tabeli? Zaokrąglij odpowiedź do dwóch miejsc po przecinku.",
        "data": pd.DataFrame({
            "minimum_sleep_hours": [round(df["sleep_hours"].min(), 2)]
        })
    },
    {
        "task_id": "E6",
        "difficulty": "easy",
        "language": "PL",
        "question": "Który kraj ma najwyższą średnią wartość instagram_minutes_daily w podanej tabeli?",
        "data": df.groupby("country", as_index=False)["instagram_minutes_daily"]
            .mean()
            .sort_values("instagram_minutes_daily", ascending=False)
            .head(30)
    }
]

In [ ]:
def generate_response(messages, max_new_tokens=100):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Изменили тут: temperature=0.0 убран
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.time()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip(), round(end_time - start_time, 3)

In [ ]:
results_pl = []

for task in simple_tasks_pl:
    table_text = task["data"].to_string(index=False)

    messages = [
        {
            "role": "system",
            "content": (
                "Jesteś analitykiem danych. "
                "Korzystaj wyłącznie z podanej tabeli. "
                "Zwróć tylko końcową odpowiedź. "
                "Nie wyjaśniaj toku rozumowania."
            )
        },
        {
            "role": "user",
            "content": f"""
Oto tabela:

{table_text}

Zadanie:
{task["question"]}
"""
        }
    ]

    answer, response_time = generate_response(messages)

    results_pl.append({
        "model": "Bielik-11B",
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "language": task["language"],
        "question": task["question"],
        "model_answer": answer,
        "correct_answer": correct_answers[task["task_id"]],
        "response_time_sec": response_time
    })

results_pl_df = pd.DataFrame(results_pl)
results_pl_df

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

,model,task_id,difficulty,language,question,model_answer,correct_answer,response_time_sec
0,Bielik-11B,E1,easy,PL,Który kraj ma najwyższą wartość addiction_scor...,Ghana.,Ghana,0.742
1,Bielik-11B,E2,easy,PL,Jaka jest średnia wartość addiction_score w po...,58.55,58.55,0.827
2,Bielik-11B,E3,easy,PL,W którym roku średnia wartość tiktok_minutes_d...,2015,2044,0.754
3,Bielik-11B,E4,easy,PL,Ile obserwacji ma addiction_level równy High?,8680,8680,0.693
4,Bielik-11B,E5,easy,PL,Jaka jest minimalna wartość sleep_hours w poda...,3.00,3.0,0.694
5,Bielik-11B,E6,easy,PL,Który kraj ma najwyższą średnią wartość instag...,UK,UK,0.303


Russian prompt

In [ ]:
simple_tasks_ru = [
    {
        "task_id": "E1",
        "difficulty": "easy",
        "language": "RU",
        "question": "Какая страна имеет самое высокое значение addiction_score в предоставленной таблице?",
        "data": df[["country", "addiction_score"]]
            .sort_values("addiction_score", ascending=False)
            .head(30)
    },
    {
        "task_id": "E2",
        "difficulty": "easy",
        "language": "RU",
        "question": "Каково среднее значение addiction_score в предоставленной таблице? Округли ответ до двух знаков после запятой.",
        "data": pd.DataFrame({
            "average_addiction_score": [round(df["addiction_score"].mean(), 2)]
        })
    },
    {
        "task_id": "E3",
        "difficulty": "easy",
        "language": "RU",
        "question": "В каком году среднее значение tiktok_minutes_daily было самым высоким в предоставленной таблице?",
        "data": df.groupby("year", as_index=False)["tiktok_minutes_daily"]
            .mean()
            .sort_values("tiktok_minutes_daily", ascending=False)
    },
    {
        "task_id": "E4",
        "difficulty": "easy",
        "language": "RU",
        "question": "Сколько наблюдений имеют значение addiction_level, равное High?",
        "data": pd.DataFrame({
            "addiction_level": ["High"],
            "number_of_observations": [int((df["addiction_level"] == "High").sum())]
        })
    },
    {
        "task_id": "E5",
        "difficulty": "easy",
        "language": "RU",
        "question": "Каково минимальное значение sleep_hours в предоставленной таблице? Округли ответ до двух знаков после запятой.",
        "data": pd.DataFrame({
            "minimum_sleep_hours": [round(df["sleep_hours"].min(), 2)]
        })
    },
    {
        "task_id": "E6",
        "difficulty": "easy",
        "language": "RU",
        "question": "Какая страна имеет самое высокое среднее значение instagram_minutes_daily в предоставленной таблице?",
        "data": df.groupby("country", as_index=False)["instagram_minutes_daily"]
            .mean()
            .sort_values("instagram_minutes_daily", ascending=False)
            .head(30)
    }
]

In [ ]:
def generate_response(messages, max_new_tokens=100):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Изменили тут: temperature=0.0 убран
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.time()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip(), round(end_time - start_time, 3)

In [ ]:
results_ru = []
for task in simple_tasks_ru:
    table_text = task["data"].to_string(index=False)
    messages = [                          # <-- must be indented inside loop
        {
            "role": "system",
            "content": (
                "Ты аналитик данных. "
                "Используй только предоставленную таблицу. "
                "Верни только итоговый ответ. "
                "Не объясняй ход рассуждений."
            )
        },
        {
            "role": "user",
            "content": f"""
Вот таблица:
{table_text}

Задание:
{task["question"]}
"""
        }
    ]
    answer, response_time = generate_response(messages)
    results_ru.append({                   # <-- must be indented inside loop
        "model": "Bielik-11B",
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "language": task["language"],
        "question": task["question"],
        "model_answer": answer,
        "correct_answer": correct_answers[task["task_id"]],
        "response_time_sec": response_time
    })

results_ru_df = pd.DataFrame(results_ru)
results_ru_df

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

,model,task_id,difficulty,language,question,model_answer,correct_answer,response_time_sec
0,Bielik-11B,E1,easy,RU,Какая страна имеет самое высокое значение addi...,Ghana,Ghana,0.610
1,Bielik-11B,E2,easy,RU,Каково среднее значение addiction_score в пред...,58.55,58.55,0.824
2,Bielik-11B,E3,easy,RU,В каком году среднее значение tiktok_minutes_d...,2015,2044,0.752
3,Bielik-11B,E4,easy,RU,Сколько наблюдений имеют значение addiction_le...,8680,8680,0.698
4,Bielik-11B,E5,easy,RU,Каково минимальное значение sleep_hours в пред...,3.00,3.0,0.701
5,Bielik-11B,E6,easy,RU,Какая страна имеет самое высокое среднее значе...,UK,UK,0.315


**6 medium level tasks**

English prompt

In [ ]:
# =========================
# MEDIUM TASKS — Bielik
# =========================

medium_correct_answers = {
    "M1": round(
        (
            df[df["year"] == 2030]["tiktok_minutes_daily"].mean()
            - df[df["year"] == 2020]["tiktok_minutes_daily"].mean()
        )
        / df[df["year"] == 2020]["tiktok_minutes_daily"].mean()
        * 100,
        2
    ),

    "M2": (
        df.groupby("country")[["tiktok_minutes_daily", "instagram_minutes_daily"]]
        .mean()
        .assign(diff=lambda x: abs(x["tiktok_minutes_daily"] - x["instagram_minutes_daily"]))
        .sort_values("diff", ascending=False)
        .index[0]
    ),

    "M3": (
        df.groupby("country")["addiction_score"]
        .mean()
        .idxmax()
    ),

    "M4": round(
        (df["sleep_quality_index"] < df["sleep_quality_index"].mean()).mean() * 100,
        2
    ),

    "M5": round(
        df[df["addiction_level"] == "High"]["MHRI"].mean()
        - df[df["addiction_level"] == "Low"]["MHRI"].mean(),
        2
    ),

    "M6": (
        df.groupby("country")["dopamine_dependency_score"]
        .mean()
        .sort_values(ascending=False)
        .head(3)
        .index.tolist()
    )
}

medium_correct_answers

{'M1': np.float64(4.45),
 'M2': 'Maldives',
 'M3': 'Tanzania',
 'M4': np.float64(49.99),
 'M5': np.float64(6.82),
 'M6': ['Tanzania', 'Argentina', 'Malaysia']}

In [ ]:
medium_tasks_en = [
    {
        "task_id": "M1",
        "difficulty": "medium",
        "language": "EN",
        "question": "By what percentage did the average tiktok_minutes_daily increase or decrease between 2020 and 2030? Round the answer to two decimal places.",
        "data": df[df["year"].isin([2020, 2030])]
            .groupby("year", as_index=False)["tiktok_minutes_daily"]
            .mean()
    },
    {
        "task_id": "M2",
        "difficulty": "medium",
        "language": "EN",
        "question": "Which country has the largest absolute difference between average tiktok_minutes_daily and average instagram_minutes_daily?",
        "data": df.groupby("country", as_index=False)[["tiktok_minutes_daily", "instagram_minutes_daily"]]
            .mean()
            .assign(diff=lambda x: abs(x["tiktok_minutes_daily"] - x["instagram_minutes_daily"]))
            .sort_values("diff", ascending=False)
            .head(30)
    },
    {
        "task_id": "M3",
        "difficulty": "medium",
        "language": "EN",
        "question": "Which country has the highest average addiction_score?",
        "data": df.groupby("country", as_index=False)["addiction_score"]
            .mean()
            .sort_values("addiction_score", ascending=False)
            .head(30)
    },
    {
        "task_id": "M4",
        "difficulty": "medium",
        "language": "EN",
        "question": "What percentage of observations have sleep_quality_index below the overall average? Round the answer to two decimal places.",
        "data": pd.DataFrame({
            "overall_average_sleep_quality_index": [df["sleep_quality_index"].mean()],
            "observations_below_average": [(df["sleep_quality_index"] < df["sleep_quality_index"].mean()).sum()],
            "total_observations": [len(df)],
            "percentage_below_average": [round((df["sleep_quality_index"] < df["sleep_quality_index"].mean()).mean() * 100, 2)]
        })
    },
    {
        "task_id": "M5",
        "difficulty": "medium",
        "language": "EN",
        "question": "What is the difference between the average MHRI for High addiction_level and Low addiction_level? Round the answer to two decimal places.",
        "data": df[df["addiction_level"].isin(["High", "Low"])]
            .groupby("addiction_level", as_index=False)["MHRI"]
            .mean()
    },
    {
        "task_id": "M6",
        "difficulty": "medium",
        "language": "EN",
        "question": "Which three countries have the highest average dopamine_dependency_score? Return the country names only.",
        "data": df.groupby("country", as_index=False)["dopamine_dependency_score"]
            .mean()
            .sort_values("dopamine_dependency_score", ascending=False)
            .head(10)
    }
]

In [ ]:
def generate_response(messages, max_new_tokens=100):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Изменили тут: temperature=0.0 убран
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.time()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip(), round(end_time - start_time, 3)

In [ ]:
results_medium_en = []

for task in medium_tasks_en:
    table_text = task["data"].to_string(index=False)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a data analyst. "
                "Use only the provided table. "
                "Return only the final answer. "
                "Do not explain your reasoning."
            )
        },
        {
            "role": "user",
            "content": f"""
Here is the table:

{table_text}

Task:
{task["question"]}
"""
        }
    ]

    answer, response_time = generate_response(messages)

    results_medium_en.append({
        "model": "Bielik-11B",
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "language": task["language"],
        "question": task["question"],
        "model_answer": answer,
        "correct_answer": medium_correct_answers[task["task_id"]],
        "response_time_sec": response_time
    })

results_medium_en_df = pd.DataFrame(results_medium_en)
results_medium_en_df

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

,model,task_id,difficulty,language,question,model_answer,correct_answer,response_time_sec
0,Bielik-11B,M1,medium,EN,By what percentage did the average tiktok_minu...,3.42,4.45,0.708
1,Bielik-11B,M2,medium,EN,Which country has the largest absolute differe...,Mozambique.,Maldives,0.918
2,Bielik-11B,M3,medium,EN,Which country has the highest average addictio...,Tanzania.,Tanzania,0.703
3,Bielik-11B,M4,medium,EN,What percentage of observations have sleep_qua...,49.99,49.99,0.827
4,Bielik-11B,M5,medium,EN,What is the difference between the average MHR...,6.82,6.82,0.697
5,Bielik-11B,M6,medium,EN,Which three countries have the highest average...,"Tanzania, Argentina, Malaysia.","[Tanzania, Argentina, Malaysia]",1.210


polish prompt

In [ ]:
medium_tasks_pl = [
    {
        "task_id": "M1",
        "difficulty": "medium",
        "language": "PL",
        "question": "O ile procent wzrosła lub spadła średnia wartość tiktok_minutes_daily między 2020 a 2030 rokiem? Zaokrąglij odpowiedź do dwóch miejsc po przecinku.",
        "data": df[df["year"].isin([2020, 2030])]
                    .groupby("year", as_index=False)["tiktok_minutes_daily"]
                    .mean()
    },
    {
        "task_id": "M2",
        "difficulty": "medium",
        "language": "PL",
        "question": "Który kraj ma największą bezwzględną różnicę między średnią wartością tiktok_minutes_daily a średnią wartością instagram_minutes_daily?",
        "data": df.groupby("country", as_index=False)[["tiktok_minutes_daily", "instagram_minutes_daily"]]
                    .mean()
                    .assign(diff=lambda x: abs(x["tiktok_minutes_daily"] - x["instagram_minutes_daily"]))
                    .sort_values("diff", ascending=False)
                    .head(30)
    },
    {
        "task_id": "M3",
        "difficulty": "medium",
        "language": "PL",
        "question": "Który kraj ma najwyższą średnią wartość addiction_score?",
        "data": df.groupby("country", as_index=False)["addiction_score"]
                    .mean()
                    .sort_values("addiction_score", ascending=False)
                    .head(30)
    },
    {
        "task_id": "M4",
        "difficulty": "medium",
        "language": "PL",
        "question": "Jaki procent obserwacji ma wartość sleep_quality_index poniżej ogólnej średniej? Zaokrąglij odpowiedź do dwóch miejsc po przecinku.",
        "data": pd.DataFrame({
            "overall_average_sleep_quality_index": [df["sleep_quality_index"].mean()],
            "observations_below_average": [(df["sleep_quality_index"] < df["sleep_quality_index"].mean()).sum()],
            "total_observations": [len(df)],
            "percentage_below_average": [round((df["sleep_quality_index"] < df["sleep_quality_index"].mean()).mean() * 100, 2)]
        })
    },
    {
        "task_id": "M5",
        "difficulty": "medium",
        "language": "PL",
        "question": "Jaka jest różnica między średnią wartością MHRI dla poziomu uzależnienia High a Low? Zaokrąglij odpowiedź do dwóch miejsc po przecinku.",
        "data": df[df["addiction_level"].isin(["High", "Low"])]
                    .groupby("addiction_level", as_index=False)["MHRI"]
                    .mean()
    },
    {
        "task_id": "M6",
        "difficulty": "medium",
        "language": "PL",
        "question": "Które trzy kraje mają najwyższą średnią wartość dopamine_dependency_score? Podaj tylko nazwy krajów.",
        "data": df.groupby("country", as_index=False)["dopamine_dependency_score"]
                    .mean()
                    .sort_values("dopamine_dependency_score", ascending=False)
                    .head(10)
    }
]

In [ ]:
def generate_response(messages, max_new_tokens=100):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Изменили тут: temperature=0.0 убран
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.time()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip(), round(end_time - start_time, 3)

In [ ]:
results_medium_pl = []
for task in medium_tasks_pl:
    table_text = task["data"].to_string(index=False)
    messages = [
        {
            "role": "system",
            "content": (
                "Jesteś analitykiem danych. "
                "Korzystaj wyłącznie z podanej tabeli. "
                "Zwróć tylko końcową odpowiedź. "
                "Nie wyjaśniaj toku rozumowania."
            )
        },
        {
            "role": "user",
            "content": f"Oto tabela:\n\n{table_text}\n\nZadanie:\n{task['question']}"
        }
    ]
    answer, response_time = generate_response(messages)
    results_medium_pl.append({
        "model": "Bielik-11B",
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "language": task["language"],
        "question": task["question"],
        "model_answer": answer,
        "correct_answer": medium_correct_answers[task["task_id"]],
        "response_time_sec": response_time
    })

results_medium_pl_df = pd.DataFrame(results_medium_pl)
results_medium_pl_df

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

,model,task_id,difficulty,language,question,model_answer,correct_answer,response_time_sec
0,Bielik-11B,M1,medium,PL,O ile procent wzrosła lub spadła średnia warto...,4.72%,4.45,0.832
1,Bielik-11B,M2,medium,PL,Który kraj ma największą bezwzględną różnicę m...,Maldives.,Maldives,0.776
2,Bielik-11B,M3,medium,PL,Który kraj ma najwyższą średnią wartość addict...,Tanzania.,Tanzania,0.695
3,Bielik-11B,M4,medium,PL,Jaki procent obserwacji ma wartość sleep_quali...,49.99,49.99,0.826
4,Bielik-11B,M5,medium,PL,Jaka jest różnica między średnią wartością MHR...,6.82,6.82,0.694
5,Bielik-11B,M6,medium,PL,Które trzy kraje mają najwyższą średnią wartoś...,"Tanzania, Argentina, Malaysia.","[Tanzania, Argentina, Malaysia]",1.225


Russian prompt

In [ ]:
medium_tasks_ru = [
    {
        "task_id": "M1",
        "difficulty": "medium",
        "language": "RU",
        "question": "На сколько процентов увеличилось или уменьшилось среднее значение tiktok_minutes_daily между 2020 и 2030 годами? Округли ответ до двух знаков после запятой.",
        "data": df[df["year"].isin([2020, 2030])]
                    .groupby("year", as_index=False)["tiktok_minutes_daily"]
                    .mean()
    },
    {
        "task_id": "M2",
        "difficulty": "medium",
        "language": "RU",
        "question": "Какая страна имеет наибольшую абсолютную разницу между средним значением tiktok_minutes_daily и средним значением instagram_minutes_daily?",
        "data": df.groupby("country", as_index=False)[["tiktok_minutes_daily", "instagram_minutes_daily"]]
                    .mean()
                    .assign(diff=lambda x: abs(x["tiktok_minutes_daily"] - x["instagram_minutes_daily"]))
                    .sort_values("diff", ascending=False)
                    .head(30)
    },
    {
        "task_id": "M3",
        "difficulty": "medium",
        "language": "RU",
        "question": "Какая страна имеет наибольшее среднее значение addiction_score?",
        "data": df.groupby("country", as_index=False)["addiction_score"]
                    .mean()
                    .sort_values("addiction_score", ascending=False)
                    .head(30)
    },
    {
        "task_id": "M4",
        "difficulty": "medium",
        "language": "RU",
        "question": "Какой процент наблюдений имеет значение sleep_quality_index ниже общего среднего? Округли ответ до двух знаков после запятой.",
        "data": pd.DataFrame({
            "overall_average_sleep_quality_index": [df["sleep_quality_index"].mean()],
            "observations_below_average": [(df["sleep_quality_index"] < df["sleep_quality_index"].mean()).sum()],
            "total_observations": [len(df)],
            "percentage_below_average": [round((df["sleep_quality_index"] < df["sleep_quality_index"].mean()).mean() * 100, 2)]
        })
    },
    {
        "task_id": "M5",
        "difficulty": "medium",
        "language": "RU",
        "question": "Какова разница между средним значением MHRI для уровня зависимости High и Low? Округли ответ до двух знаков после запятой.",
        "data": df[df["addiction_level"].isin(["High", "Low"])]
                    .groupby("addiction_level", as_index=False)["MHRI"]
                    .mean()
    },
    {
        "task_id": "M6",
        "difficulty": "medium",
        "language": "RU",
        "question": "Какие три страны имеют наибольшее среднее значение dopamine_dependency_score? Укажи только названия стран.",
        "data": df.groupby("country", as_index=False)["dopamine_dependency_score"]
                    .mean()
                    .sort_values("dopamine_dependency_score", ascending=False)
                    .head(10)
    }
]

In [ ]:
def generate_response(messages, max_new_tokens=100):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Изменили тут: temperature=0.0 убран
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.time()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip(), round(end_time - start_time, 3)

In [ ]:
results_medium_ru = []
for task in medium_tasks_ru:
    table_text = task["data"].to_string(index=False)
    messages = [
        {
            "role": "system",
            "content": (
                "Ты аналитик данных. "
                "Используй только предоставленную таблицу. "
                "Верни только итоговый ответ. "
                "Не объясняй ход рассуждений."
            )
        },
        {
            "role": "user",
            "content": f"Вот таблица:\n\n{table_text}\n\nЗадание:\n{task['question']}"
        }
    ]
    answer, response_time = generate_response(messages)
    results_medium_ru.append({
        "model": "Bielik-11B",
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "language": task["language"],
        "question": task["question"],
        "model_answer": answer,
        "correct_answer": medium_correct_answers[task["task_id"]],
        "response_time_sec": response_time
    })

results_medium_ru_df = pd.DataFrame(results_medium_ru)
results_medium_ru_df

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

,model,task_id,difficulty,language,question,model_answer,correct_answer,response_time_sec
0,Bielik-11B,M1,medium,RU,На сколько процентов увеличилось или уменьшило...,10.47%,4.45,0.949
1,Bielik-11B,M2,medium,RU,Какая страна имеет наибольшую абсолютную разни...,Maldives.,Maldives,0.788
2,Bielik-11B,M3,medium,RU,Какая страна имеет наибольшее среднее значение...,Tanzania.,Tanzania,0.726
3,Bielik-11B,M4,medium,RU,Какой процент наблюдений имеет значение sleep_...,49.99,49.99,0.850
4,Bielik-11B,M5,medium,RU,Какова разница между средним значением MHRI дл...,6.82,6.82,0.701
5,Bielik-11B,M6,medium,RU,Какие три страны имеют наибольшее среднее знач...,"Tanzania, Argentina, Malaysia.","[Tanzania, Argentina, Malaysia]",1.221


**4 hard level tasks**

English prompt

In [ ]:
# =========================
# HARD TASKS — Bielik — EN
# =========================

import numpy as np
from sklearn.linear_model import LinearRegression

In [ ]:
# H1
sleep_mean = df["sleep_hours"].mean()

h1_base = df[
    (df["age"] < 25) &
    (df["sleep_hours"] < sleep_mean)
]

h1_answer = (
    h1_base.groupby("country")["addiction_score"]
    .mean()
    .idxmax()
)

# H2
dopamine_median = df["dopamine_dependency_score"].median()
mhri_median = df["MHRI"].median()

h2_base = df[
    (df["dopamine_dependency_score"] > dopamine_median) &
    (df["MHRI"] > mhri_median)
]

h2_answer = (
    h2_base.groupby("country")["addiction_score"]
    .mean()
    .idxmax()
)

# H3
risk_df = df.copy()
risk_df["combined_risk_score"] = (
    risk_df["addiction_score"]
    + risk_df["dopamine_dependency_score"]
    - risk_df["sleep_hours"]
)

h3_answer = (
    risk_df.groupby("country")["combined_risk_score"]
    .mean()
    .idxmax()
)

# H4
h4_base = df[df["sleep_hours"] < sleep_mean]

h4_answer_df = (
    h4_base.groupby(["country", "age"], as_index=False)["addiction_score"]
    .mean()
    .sort_values("addiction_score", ascending=False)
    .head(3)
)

h4_answer = [
    f"{row['country']}-{int(row['age'])}"
    for _, row in h4_answer_df.iterrows()
]

hard_correct_answers = {
    "H1": h1_answer,
    "H2": h2_answer,
    "H3": h3_answer,
    "H4": h4_answer
}

hard_correct_answers

{'H1': 'Morocco',
 'H2': 'Pakistan',
 'H3': 'Tanzania',
 'H4': ['Bulgaria-32', 'Vietnam-58', 'Paraguay-41']}

In [ ]:
hard_tasks_en = [
    {
        "task_id": "H1",
        "difficulty": "hard",
        "language": "EN",
        "question": (
            "Among users younger than 25 years old and with sleep_hours below "
            "the overall dataset average, which country has the highest average "
            "addiction_score? Return only the country name."
        ),
        "data": (
            h1_base.groupby("country", as_index=False)["addiction_score"]
            .mean()
            .sort_values("addiction_score", ascending=False)
            .head(30)
        )
    },
    {
        "task_id": "H2",
        "difficulty": "hard",
        "language": "EN",
        "question": (
            "Among users whose dopamine_dependency_score and MHRI are both above "
            "their respective dataset medians, which country has the highest average "
            "addiction_score? Return only the country name."
        ),
        "data": (
            h2_base.groupby("country", as_index=False)["addiction_score"]
            .mean()
            .sort_values("addiction_score", ascending=False)
            .head(30)
        )
    },
    {
        "task_id": "H3",
        "difficulty": "hard",
        "language": "EN",
        "question": (
            "Using the combined digital risk score defined as addiction_score + "
            "dopamine_dependency_score - sleep_hours, which country has the highest "
            "average combined digital risk? Return only the country name."
        ),
        "data": (
            risk_df.groupby("country", as_index=False)["combined_risk_score"]
            .mean()
            .sort_values("combined_risk_score", ascending=False)
            .head(30)
        )
    },
    {
        "task_id": "H4",
        "difficulty": "hard",
        "language": "EN",
        "question": (
            "Rank the three most vulnerable user groups, defined as country-age "
            "combinations, according to average addiction_score, considering only "
            "groups with below-average sleep_hours. Return only the top three "
            "country-age groups."
        ),
        "data": (
            h4_base.groupby(["country", "age"], as_index=False)["addiction_score"]
            .mean()
            .sort_values("addiction_score", ascending=False)
            .head(20)
        )
    }
]

In [ ]:
def generate_response(messages, max_new_tokens=100):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Изменили тут: temperature=0.0 убран
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.time()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip(), round(end_time - start_time, 3)

In [ ]:
results_hard_en = []

for task in hard_tasks_en:
    table_text = task["data"].to_string(index=False)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a data analyst. "
                "Use only the provided table. "
                "Return only the final answer. "
                "Do not explain your reasoning."
            )
        },
        {
            "role": "user",
            "content": f"""
Here is the table:

{table_text}

Task:
{task["question"]}
"""
        }
    ]

    answer, response_time = generate_response(messages)

    results_hard_en.append({
        "model": "Bielik-11B",
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "language": task["language"],
        "question": task["question"],
        "model_answer": answer,
        "correct_answer": hard_correct_answers[task["task_id"]],
        "response_time_sec": response_time
    })

results_hard_en_df = pd.DataFrame(results_hard_en)
results_hard_en_df

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

,model,task_id,difficulty,language,question,model_answer,correct_answer,response_time_sec
0,Bielik-11B,H1,hard,EN,Among users younger than 25 years old and with...,Morocco,Morocco,0.731
1,Bielik-11B,H2,hard,EN,Among users whose dopamine_dependency_score an...,Norway,Pakistan,0.574
2,Bielik-11B,H3,hard,EN,Using the combined digital risk score defined ...,Tanzania,Tanzania,0.571
3,Bielik-11B,H4,hard,EN,"Rank the three most vulnerable user groups, de...","Belgium-53, UAE-35, USA-14.","[Bulgaria-32, Vietnam-58, Paraguay-41]",2.521


Polish prompt

In [ ]:
hard_tasks_pl = [
    {
        "task_id": "H1",
        "difficulty": "hard",
        "language": "PL",
        "question": (
            "Wśród użytkowników poniżej 25 roku życia oraz z wartością sleep_hours "
            "niższą niż średnia w całym zbiorze danych, który kraj ma najwyższą "
            "średnią wartość addiction_score? Zwróć tylko nazwę kraju."
        ),
        "data": (
            h1_base.groupby("country", as_index=False)["addiction_score"]
            .mean()
            .sort_values("addiction_score", ascending=False)
            .head(30)
        )
    },
    {
        "task_id": "H2",
        "difficulty": "hard",
        "language": "PL",
        "question": (
            "Wśród użytkowników, których dopamine_dependency_score oraz MHRI są wyższe "
            "od odpowiednich median w zbiorze danych, który kraj ma najwyższą średnią "
            "wartość addiction_score? Zwróć tylko nazwę kraju."
        ),
        "data": (
            h2_base.groupby("country", as_index=False)["addiction_score"]
            .mean()
            .sort_values("addiction_score", ascending=False)
            .head(30)
        )
    },
    {
        "task_id": "H3",
        "difficulty": "hard",
        "language": "PL",
        "question": (
            "Korzystając ze złożonego wskaźnika ryzyka cyfrowego zdefiniowanego jako "
            "addiction_score + dopamine_dependency_score - sleep_hours, który kraj ma "
            "najwyższą średnią wartość tego wskaźnika? Zwróć tylko nazwę kraju."
        ),
        "data": (
            risk_df.groupby("country", as_index=False)["combined_risk_score"]
            .mean()
            .sort_values("combined_risk_score", ascending=False)
            .head(30)
        )
    },
    {
        "task_id": "H4",
        "difficulty": "hard",
        "language": "EN",
        "question": (
            "Uszereguj trzy najbardziej narażone grupy użytkowników, zdefiniowane jako "
            "kombinacje country-age, według średniej wartości addiction_score, uwzględniając "
            "tylko grupy z sleep_hours poniżej średniej. Zwróć tylko trzy najlepsze grupy country-age."
        ),
        "data": (
            h4_base.groupby(["country", "age"], as_index=False)["addiction_score"]
            .mean()
            .sort_values("addiction_score", ascending=False)
            .head(20)
        )
    }
]

In [ ]:
def generate_response(messages, max_new_tokens=100):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Изменили тут: temperature=0.0 убран
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.time()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip(), round(end_time - start_time, 3)

In [ ]:
results_hard_pl = []

for task in hard_tasks_pl:
    table_text = task["data"].to_string(index=False)

    messages = [
        {
            "role": "system",
            "content": (
                "Jesteś analitykiem danych. "
                "Korzystaj wyłącznie z podanej tabeli. "
                "Zwróć tylko końcową odpowiedź. "
                "Nie wyjaśniaj toku rozumowania."
            )
        },
        {
            "role": "user",
            "content": f"""
Oto tabela:

{table_text}

Zadanie:
{task["question"]}
"""
        }
    ]

    answer, response_time = generate_response(messages)

    results_hard_pl.append({
        "model": "Bielik-11B",
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "language": task["language"],
        "question": task["question"],
        "model_answer": answer,
        "correct_answer": hard_correct_answers[task["task_id"]],
        "response_time_sec": response_time
    })

results_hard_pl_df = pd.DataFrame(results_hard_pl)
results_hard_pl_df

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

,model,task_id,difficulty,language,question,model_answer,correct_answer,response_time_sec
0,Bielik-11B,H1,hard,PL,Wśród użytkowników poniżej 25 roku życia oraz ...,Morocco,Morocco,0.729
1,Bielik-11B,H2,hard,PL,"Wśród użytkowników, których dopamine_dependenc...",Norway,Pakistan,0.574
2,Bielik-11B,H3,hard,PL,Korzystając ze złożonego wskaźnika ryzyka cyfr...,Tanzania,Tanzania,0.573
3,Bielik-11B,H4,hard,EN,Uszereguj trzy najbardziej narażone grupy użyt...,| country | age | addiction_score |\n|-----...,"[Bulgaria-32, Vietnam-58, Paraguay-41]",11.135


Russian prompt

In [ ]:
hard_tasks_ru = [
    {
        "task_id": "H1",
        "difficulty": "hard",
        "language": "RU",
        "question": (
            "Среди пользователей младше 25 лет и с количеством sleep_hours ниже "
            "среднего значения по всему датасету, какая страна имеет самое высокое "
            "среднее значение addiction_score? Верни только название страны."
        ),
        "data": (
            h1_base.groupby("country", as_index=False)["addiction_score"]
            .mean()
            .sort_values("addiction_score", ascending=False)
            .head(30)
        )
    },
    {
        "task_id": "H2",
        "difficulty": "hard",
        "language": "RU",
        "question": (
            "Среди пользователей, у которых dopamine_dependency_score и MHRI выше "
            "соответствующих медианных значений по датасету, какая страна имеет самое "
            "высокое среднее значение addiction_score? Верни только название страны."

        ),
        "data": (
            h2_base.groupby("country", as_index=False)["addiction_score"]
            .mean()
            .sort_values("addiction_score", ascending=False)
            .head(30)
        )
    },
    {
        "task_id": "H3",
        "difficulty": "hard",
        "language": "RU",
        "question": (
            "Используя комбинированный цифровой риск, определённый как "
            "addiction_score + dopamine_dependency_score - sleep_hours, какая страна "
            "имеет самый высокий средний комбинированный цифровой риск? Верни только название страны."
        ),
        "data": (
            risk_df.groupby("country", as_index=False)["combined_risk_score"]
            .mean()
            .sort_values("combined_risk_score", ascending=False)
            .head(30)
        )
    },
    {
        "task_id": "H4",
        "difficulty": "hard",
        "language": "RU",
        "question": (
            "Ранжируй три наиболее уязвимые группы пользователей, определённые как "
            "комбинации country-age, по среднему значению addiction_score, учитывая "
            "только группы с sleep_hours ниже среднего значения. Верни только три лучшие группы country-age."
        ),
        "data": (
            h4_base.groupby(["country", "age"], as_index=False)["addiction_score"]
            .mean()
            .sort_values("addiction_score", ascending=False)
            .head(20)
        )
    }
]

In [ ]:
def generate_response(messages, max_new_tokens=100):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Изменили тут: temperature=0.0 убран
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.time()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip(), round(end_time - start_time, 3)

In [ ]:
results_hard_ru = []

for task in hard_tasks_ru:
    table_text = task["data"].to_string(index=False)

    messages = [
        {
            "role": "system",
            "content": (
                "Ты аналитик данных. "
                "Используй только предоставленную таблицу. "
                "Верни только итоговый ответ. "
                "Не объясняй ход рассуждений."
            )
        },
        {
            "role": "user",
            "content": f"""
Вот таблица:

{table_text}

Задание:
{task["question"]}
"""
        }
    ]

    answer, response_time = generate_response(messages)

    results_hard_ru.append({
        "model": "Bielik-11B",
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "language": task["language"],
        "question": task["question"],
        "model_answer": answer,
        "correct_answer": hard_correct_answers[task["task_id"]],
        "response_time_sec": response_time
    })

results_hard_ru_df = pd.DataFrame(results_hard_ru)
results_hard_ru_df

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

,model,task_id,difficulty,language,question,model_answer,correct_answer,response_time_sec
0,Bielik-11B,H1,hard,RU,Среди пользователей младше 25 лет и с количест...,Morocco,Morocco,0.741
1,Bielik-11B,H2,hard,RU,"Среди пользователей, у которых dopamine_depend...",**UAE**,Pakistan,0.708
2,Bielik-11B,H3,hard,RU,"Используя комбинированный цифровой риск, опред...",Tanzania,Tanzania,0.577
3,Bielik-11B,H4,hard,RU,Ранжируй три наиболее уязвимые группы пользова...,**Ответ:**\n\n```\ncountry age addiction_sco...,"[Bulgaria-32, Vietnam-58, Paraguay-41]",9.321


**Forecast tasks**

English prompt

In [ ]:
# =========================
# FORECAST TASKS — Bielik — EN
# WITH PREDICTION + JUSTIFICATION
# =========================

import pandas as pd
import numpy as np
import re
from sklearn.linear_model import LinearRegression

In [ ]:
# 1. Prepare yearly data

yearly_addiction = (
    df.groupby("year", as_index=False)["addiction_score"]
    .mean()
    .sort_values("year")
)

train_forecast = yearly_addiction[yearly_addiction["year"] <= 2040].copy()

X_train = train_forecast[["year"]]
y_train = train_forecast["addiction_score"]

forecast_model = LinearRegression()
forecast_model.fit(X_train, y_train)

# Correct answer for F1
f1_answer = round(
    float(forecast_model.predict(pd.DataFrame({"year": [2045]}))[0]),
    2
)

# Correct answer for F2
pred_2040 = round(
    float(forecast_model.predict(pd.DataFrame({"year": [2040]}))[0]),
    2
)

pred_2050 = round(
    float(forecast_model.predict(pd.DataFrame({"year": [2050]}))[0]),
    2
)

f2_answer = "Increase" if pred_2050 > pred_2040 else "Decrease"

forecast_correct_answers = {
    "F1": f1_answer,
    "F2": f2_answer
}

forecast_correct_answers

{'F1': 58.37, 'F2': 'Decrease'}

In [ ]:
# 2. Forecast tasks in English

forecast_tasks_en = [
    {
        "task_id": "F1",
        "difficulty": "forecast",
        "language": "EN",
        "question": (
            "Using a simple linear trend based on the data from 2015 to 2040, "
            "predict the average addiction_score for year 2045. "
            "Return your answer in this exact format:\n"
            "Prediction: <number>\n"
            "Justification: <one short sentence>"
        ),
        "data": train_forecast
    },
    {
        "task_id": "F2",
        "difficulty": "forecast",
        "language": "EN",
        "question": (
            "Using a simple linear trend based on the data from 2015 to 2040, "
            "determine whether the average addiction_score in 2050 is expected "
            "to increase or decrease compared to 2040. "
            "Return your answer in this exact format:\n"
            "Prediction: <Increase or Decrease>\n"
            "Justification: <one short sentence>"
        ),
        "data": train_forecast
    }
]

In [ ]:
def generate_response(messages, max_new_tokens=150):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Изменили тут: temperature=0.0 убран
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.time()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip(), round(end_time - start_time, 3)

In [ ]:
# 3. Run forecast tasks

results_forecast_en = []

for task in forecast_tasks_en:
    table_text = task["data"].to_string(index=False)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a data analyst. "
                "Use only the provided table. "
                "Use a simple linear trend when forecasting. "
                "Return the answer exactly in the requested format. "
                "Do not add anything else."
            )
        },
        {
            "role": "user",
            "content": f"""
Here is the table:

{table_text}

Task:
{task["question"]}
"""
        }
    ]

    answer, response_time = generate_response(messages)

    results_forecast_en.append({
        "model": "Qwen3-8B",
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "language": task["language"],
        "question": task["question"],
        "model_answer": answer,
        "correct_answer": forecast_correct_answers[task["task_id"]],
        "response_time_sec": response_time
    })

results_forecast_en_df = pd.DataFrame(results_forecast_en)
results_forecast_en_df

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


,model,task_id,difficulty,language,question,model_answer,correct_answer,response_time_sec
0,Qwen3-8B,F1,forecast,EN,Using a simple linear trend based on the data ...,Prediction: 58.33\nJustification: Linear trend...,58.37,3.812
1,Qwen3-8B,F2,forecast,EN,Using a simple linear trend based on the data ...,Prediction: Increase\nJustification: The trend...,Decrease,3.704


Polish prompt

In [ ]:
forecast_tasks_pl = [
    {
        "task_id": "F1",
        "difficulty": "forecast",
        "language": "PL",
        "question": (
            "Na podstawie prostego trendu liniowego wyznaczonego z danych "
            "z lat 2015–2040 przewidź średnią wartość addiction_score dla roku 2045. "
            "Zwróć odpowiedź dokładnie w następującym formacie:\n"
            "Prediction: <liczba>\n"
            "Justification: <jedno krótkie zdanie>"
        ),
        "data": train_forecast
    },
    {
        "task_id": "F2",
        "difficulty": "forecast",
        "language": "PL",
        "question": (
            "Na podstawie prostego trendu liniowego wyznaczonego z danych "
            "z lat 2015–2040 określ, czy średnia wartość addiction_score "
            "w roku 2050 będzie wyższa czy niższa niż w roku 2040. "
            "Zwróć odpowiedź dokładnie w następującym formacie:\n"
            "Prediction: <Increase lub Decrease>\n"
            "Justification: <jedno krótkie zdanie>"
        ),
        "data": train_forecast
    }
]

In [ ]:
def generate_response(messages, max_new_tokens=150):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Изменили тут: temperature=0.0 убран
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.time()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip(), round(end_time - start_time, 3)

In [ ]:
results_forecast_pl = []

for task in forecast_tasks_pl:

    table_text = task["data"].to_string(index=False)

    messages = [
        {
            "role": "system",
            "content": (
                "Jesteś analitykiem danych. "
                "Korzystaj wyłącznie z podanej tabeli. "
                "Do prognozowania użyj prostego trendu liniowego. "
                "Zwróć odpowiedź dokładnie w wymaganym formacie. "
                "Nie dodawaj nic więcej."
            )
        },
        {
            "role": "user",
            "content": f"""
Oto tabela:

{table_text}

Zadanie:
{task["question"]}
"""
        }
    ]

    answer, response_time = generate_response(messages)

    results_forecast_pl.append({
        "model": "Qwen3-8B",
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "language": task["language"],
        "question": task["question"],
        "model_answer": answer,
        "correct_answer": forecast_correct_answers[task["task_id"]],
        "response_time_sec": response_time
    })

results_forecast_pl_df = pd.DataFrame(results_forecast_pl)
results_forecast_pl_df

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


,model,task_id,difficulty,language,question,model_answer,correct_answer,response_time_sec
0,Qwen3-8B,F1,forecast,PL,Na podstawie prostego trendu liniowego wyznacz...,Prediction: 58.3\nJustification: Trend liniowy...,58.37,5.106
1,Qwen3-8B,F2,forecast,PL,Na podstawie prostego trendu liniowego wyznacz...,Prediction: Increase\nJustification: Trend lin...,Decrease,5.555


Russian prompt

In [ ]:
forecast_tasks_ru = [
    {
        "task_id": "F1",
        "difficulty": "forecast",
        "language": "RU",
        "question": (
            "Используя простой линейный тренд на основе данных за период "
            "2015–2040 годов, спрогнозируй среднее значение addiction_score "
            "для 2045 года. "
            "Верни ответ строго в следующем формате:\n"
            "Prediction: <число>\n"
            "Justification: <одно короткое предложение>"
        ),
        "data": train_forecast
    },
    {
        "task_id": "F2",
        "difficulty": "forecast",
        "language": "RU",
        "question": (
            "Используя простой линейный тренд на основе данных за период "
            "2015–2040 годов, определи, будет ли среднее значение "
            "addiction_score в 2050 году выше или ниже, чем в 2040 году. "
            "Верни ответ строго в следующем формате:\n"
            "Prediction: <Increase или Decrease>\n"
            "Justification: <одно короткое предложение>"
        ),
        "data": train_forecast
    }
]

In [ ]:
def generate_response(messages, max_new_tokens=150):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Изменили тут: temperature=0.0 убран
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.time()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip(), round(end_time - start_time, 3)

In [ ]:
results_forecast_ru = []

for task in forecast_tasks_ru:

    table_text = task["data"].to_string(index=False)

    messages = [
        {
            "role": "system",
            "content": (
                "Ты аналитик данных. "
                "Используй только предоставленную таблицу. "
                "Для прогнозирования используй простой линейный тренд. "
                "Верни ответ строго в требуемом формате. "
                "Не добавляй ничего лишнего."
            )
        },
        {
            "role": "user",
            "content": f"""
Вот таблица:

{table_text}

Задание:
{task["question"]}
"""
        }
    ]

    answer, response_time = generate_response(messages)

    results_forecast_ru.append({
        "model": "Qwen3-8B",
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "language": task["language"],
        "question": task["question"],
        "model_answer": answer,
        "correct_answer": forecast_correct_answers[task["task_id"]],
        "response_time_sec": response_time
    })

results_forecast_ru_df = pd.DataFrame(results_forecast_ru)
results_forecast_ru_df

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


,model,task_id,difficulty,language,question,model_answer,correct_answer,response_time_sec
0,Qwen3-8B,F1,forecast,RU,Используя простой линейный тренд на основе дан...,Prediction: 58.322852\nJustification: Среднее ...,58.37,4.449
1,Qwen3-8B,F2,forecast,RU,Используя простой линейный тренд на основе дан...,Prediction: Increase\nJustification: Трендовая...,Decrease,2.659
